# Exploratory Analysis – Regulatory Uncertainty LLM Index

This notebook explores the distribution of uncertainty scores and validates the index generation pipeline.

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Sample Data

In [ ]:
# Generate synthetic uncertainty scores for demonstration
np.random.seed(42)

dates = pd.date_range(start='2026-01-01', end='2026-05-11', freq='D')
n_docs = 300

# Create realistic uncertainty pattern with trend and seasonality
scores = []
for i, date in enumerate(dates):
    # Trend component
    trend = 0.4 + 0.1 * (i / len(dates))
    
    # Seasonal component
    seasonal = 0.1 * np.sin(2 * np.pi * i / 30)
    
    # Random noise
    noise = np.random.normal(0, 0.05)
    
    base_score = np.clip(trend + seasonal + noise, 0, 1)
    
    # Generate multiple documents for this date
    for j in range(3):
        doc_variation = np.random.normal(0, 0.1)
        score = np.clip(base_score + doc_variation, 0, 1)
        
        scores.append({
            'timestamp': date,
            'score': score,
            'confidence': np.random.uniform(0.7, 1.0),
            'uncertainty_level': ['LOW', 'MODERATE', 'HIGH'][int(score * 3)],
            'regulatory_domain': np.random.choice(['credit', 'market', 'operational']),
            'document_id': f'doc_{len(scores):04d}'
        })

df = pd.DataFrame(scores)
print(f"Dataset shape: {df.shape}")
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nScore statistics:")
print(df['score'].describe())

## 2. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Score distribution
axes[0, 0].hist(df['score'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Uncertainty Scores')
axes[0, 0].set_xlabel('Score')
axes[0, 0].set_ylabel('Frequency')

# Uncertainty level counts
level_counts = df['uncertainty_level'].value_counts()
axes[0, 1].bar(level_counts.index, level_counts.values, color=['green', 'orange', 'red'][:len(level_counts)])
axes[0, 1].set_title('Uncertainty Level Distribution')
axes[0, 1].set_ylabel('Count')

# By regulatory domain
domain_scores = df.groupby('regulatory_domain')['score'].mean()
axes[1, 0].bar(domain_scores.index, domain_scores.values, color='steelblue')
axes[1, 0].set_title('Average Score by Regulatory Domain')
axes[1, 0].set_ylabel('Average Score')

# Confidence vs Score scatter
axes[1, 1].scatter(df['score'], df['confidence'], alpha=0.5)
axes[1, 1].set_title('Confidence vs Score')
axes[1, 1].set_xlabel('Score')
axes[1, 1].set_ylabel('Confidence')

plt.tight_layout()
plt.show()

## 3. Temporal Analysis

In [ ]:
# Daily aggregation
daily_index = df.groupby(df['timestamp'].dt.date).agg({
    'score': ['mean', 'std', 'count'],
    'confidence': 'mean'
}).reset_index()

daily_index.columns = ['date', 'index_value', 'volatility', 'doc_count', 'avg_confidence']
daily_index['date'] = pd.to_datetime(daily_index['date'])

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(daily_index['date'], daily_index['index_value'], label='Index Value', linewidth=2, color='steelblue')
ax.fill_between(daily_index['date'], 
                 daily_index['index_value'] - daily_index['volatility'],
                 daily_index['index_value'] + daily_index['volatility'],
                 alpha=0.3, label='±1 Std Dev')

# Add EMA
ema = daily_index['index_value'].ewm(alpha=0.3).mean()
ax.plot(daily_index['date'], ema, label='EMA (α=0.3)', linewidth=2, color='orange', linestyle='--')

ax.set_title('Regulatory Uncertainty Index – Temporal Analysis')
ax.set_xlabel('Date')
ax.set_ylabel('Index Value')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nIndex Statistics:")
print(f"Mean: {daily_index['index_value'].mean():.4f}")
print(f"Std Dev: {daily_index['index_value'].std():.4f}")
print(f"Min: {daily_index['index_value'].min():.4f}")
print(f"Max: {daily_index['index_value'].max():.4f}")

## 4. Domain-Specific Analysis

In [ ]:
# Daily index by domain
daily_by_domain = df.groupby([df['timestamp'].dt.date, 'regulatory_domain'])['score'].mean().unstack()
daily_by_domain.index = pd.to_datetime(daily_by_domain.index)

fig, ax = plt.subplots(figsize=(14, 6))

for domain in daily_by_domain.columns:
    ax.plot(daily_by_domain.index, daily_by_domain[domain], marker='o', label=domain, linewidth=2)

ax.set_title('Domain-Specific Uncertainty Indices')
ax.set_xlabel('Date')
ax.set_ylabel('Index Value')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlation matrix
correlation = daily_by_domain.corr()
print(f"\nDomain Correlations:")
print(correlation)

## 5. Top Uncertainty Events

In [ ]:
# Top HIGH/CRITICAL documents
high_uncertainty = df[df['uncertainty_level'].isin(['HIGH', 'CRITICAL'])].copy()
high_uncertainty = high_uncertainty.nlargest(10, 'score')

print("Top 10 High Uncertainty Events:")
print(high_uncertainty[['timestamp', 'score', 'confidence', 'uncertainty_level', 'regulatory_domain']].to_string())

# Timeline of HIGH+ events
high_timeline = high_uncertainty.groupby('timestamp').size()

fig, ax = plt.subplots(figsize=(14, 4))
high_timeline.plot(kind='bar', ax=ax, color='red', alpha=0.7)
ax.set_title('Timeline of High Uncertainty Events')
ax.set_xlabel('Date')
ax.set_ylabel('Count of HIGH/CRITICAL Events')
plt.tight_layout()
plt.show()

## 6. Validation Checks

In [ ]:
# Autocorrelation
from scipy import stats

autocorr = daily_index['index_value'].autocorr(lag=1)
print(f"Autocorrelation (lag=1): {autocorr:.4f}")

# Data quality checks
print(f"\nData Quality Checks:")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Score range violations: {((df['score'] < 0) | (df['score'] > 1)).sum()}")
print(f"Confidence range violations: {((df['confidence'] < 0) | (df['confidence'] > 1)).sum()}")

# Distribution of document counts
print(f"\nDocuments per date:")
print(daily_index['doc_count'].describe())

## 7. Export Results

In [ ]:
# Export daily index
daily_index.to_csv('../data/processed/index_exploratory.csv', index=False)
print("Exported daily index to data/processed/index_exploratory.csv")

# Export raw scores
df.to_csv('../data/processed/scores_exploratory.csv', index=False)
print("Exported raw scores to data/processed/scores_exploratory.csv")